<a href="https://colab.research.google.com/github/vineetkrtyagi/CudaProgramming/blob/main/Vector_Add.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Tue Jun 16 10:21:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
%%writefile main.cpp
#include <iostream>
using namespace std;

int main() {
    cout << "Hello, World!" << endl;
    return 0;
}

Writing main.cpp


In [4]:
!g++ main.cpp -o main
!./main

Hello, World!


In [5]:
%%writefile vector_add.cu

#include <cuda_runtime.h>

#include <chrono>
#include <cmath>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <vector>

#define CUDA_CHECK(call)                                \
do {                                                    \
    cudaError_t err = call;                             \
    if (err != cudaSuccess) {                           \
        std::cerr << "CUDA Error: "                     \
                  << cudaGetErrorString(err)            \
                  << " at " << __FILE__ << ":"          \
                  << __LINE__ << std::endl;             \
        exit(EXIT_FAILURE);                             \
    }                                                   \
} while (0)

__global__ void add_kernel(const float* a,
                           const float* b,
                           float* c,
                           int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n)
        c[idx] = a[idx] + b[idx];
}

int main() {

    int num_elements = 1 << 20; // 1,048,576

    std::cout << "Vector Addition (N = "
              << num_elements << ")\n";

    size_t bytes = num_elements * sizeof(float);

    // Host vectors
    std::vector<float> h_a(num_elements);
    std::vector<float> h_b(num_elements);
    std::vector<float> h_c_cpu(num_elements);
    std::vector<float> h_c_gpu(num_elements);

    // Initialize input vectors
    for (int i = 0; i < num_elements; i++) {
        h_a[i] = static_cast<float>(i);
        h_b[i] = static_cast<float>(2 * i);
    }

    // Device pointers
    float *d_a = nullptr;
    float *d_b = nullptr;
    float *d_c = nullptr;

    // Allocate GPU memory
    CUDA_CHECK(cudaMalloc((void**)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_c, bytes));

    // Copy inputs to GPU
    CUDA_CHECK(cudaMemcpy(d_a, h_a.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, h_b.data(), bytes, cudaMemcpyHostToDevice));

    int threads_per_block = 256;
    int blocks = (num_elements + threads_per_block - 1) / threads_per_block;

    // CUDA events
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    CUDA_CHECK(cudaEventRecord(start));

    add_kernel<<<blocks, threads_per_block>>>(d_a, d_b, d_c, num_elements);

    // Check kernel launch
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));

    float gpu_time_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&gpu_time_ms, start, stop));

    // Copy output back
    CUDA_CHECK(cudaMemcpy(h_c_gpu.data(), d_c, bytes,
                          cudaMemcpyDeviceToHost));

    // CPU timing
    auto cpu_start = std::chrono::high_resolution_clock::now();

    for (int i = 0; i < num_elements; i++)
        h_c_cpu[i] = h_a[i] + h_b[i];

    auto cpu_end = std::chrono::high_resolution_clock::now();

    std::chrono::duration<double, std::milli>
        cpu_time = cpu_end - cpu_start;

    // Verify
    bool passed = true;

    for (int i = 0; i < num_elements; i++) {
        if (std::fabs(h_c_cpu[i] - h_c_gpu[i]) > 1e-5f) {
            passed = false;
            std::cout << "Mismatch at " << i << std::endl;
            break;
        }
    }

    std::cout << std::fixed << std::setprecision(3);
    std::cout << "GPU Kernel Time : "
              << gpu_time_ms << " ms\n";

    std::cout << "CPU Time        : "
              << cpu_time.count() << " ms\n";

    std::cout << "Verification: "
              << (passed ? "PASSED" : "FAILED")
              << std::endl;

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    return 0;
}

Writing vector_add.cu


In [6]:
!nvcc -o vector_add vector_add.cu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [12]:
!./vector_add

Vector Addition (N = 1048576)
GPU Kernel Time : 0.135 ms
CPU Time        : 7.902 ms
Verification: PASSED


In [14]:
```markdown

```cuda
%%writefile vector_add.cu
```
This is a Colab-specific magic command. It tells Colab to write the following code block into a file named `vector_add.cu`.

```cuda
#include <cuda_runtime.h>
```
Includes the core CUDA runtime API functions and types, essential for any CUDA program.

```cuda
#include <chrono>
#include <cmath>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <vector>
```
These are standard C++ headers:
*   `<chrono>`: For measuring execution time (CPU timing).
*   `<cmath>`: For mathematical functions like `std::fabs` (for floating-point absolute value).
*   `<cstdlib>`: For general utilities, including `exit(EXIT_FAILURE)`.
*   `<iomanip>`: For output formatting, like `std::setprecision`.
*   `<iostream>`: For input/output operations, like `std::cout` and `std::cerr`.
*   `<vector>`: For using `std::vector` to manage dynamic arrays on the host (CPU).

```cuda
#define CUDA_CHECK(call)                                \
do {                                                    \
    cudaError_t err = call;                             \
    if (err != cudaSuccess) {                           \
        std::cerr << "CUDA Error: "                     \
                  << cudaGetErrorString(err)            \
                  << " at " << __FILE__ << ":"          \
                  << __LINE__ << std::endl;             \
        exit(EXIT_FAILURE);                             \
    }                                                   \
} while (0)
```
This is a macro used for error checking CUDA API calls. If a `cuda` function call (represented by `call`) returns an error, it prints the error message, file name, and line number to `std::cerr` and then exits the program. This makes debugging CUDA code much easier.

```cuda
__global__ void add_kernel(const float* a,
                           const float* b,
                           float* c,
                           int n)
```
This declares a CUDA *kernel* function named `add_kernel`.
*   `__global__`: This specifier indicates that the function will be executed on the GPU by multiple threads and can be called from the CPU (host).
*   `void`: The kernel does not return a value.
*   `const float* a`, `const float* b`: Pointers to constant input arrays `a` and `b` on the device (GPU).
*   `float* c`: Pointer to the output array `c` on the device.
*   `int n`: The total number of elements in the vectors.

```cuda
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
```
Inside the kernel, this line calculates a unique global index (`idx`) for each thread.
*   `blockIdx.x`: The X-dimension index of the current thread block within the grid.
*   `blockDim.x`: The X-dimension size of each thread block (number of threads per block).
*   `threadIdx.x`: The X-dimension index of the current thread within its block.
This formula allows each thread to process a unique element of the vector.

```cuda
    if (idx < n)
        c[idx] = a[idx] + b[idx];
}
```
This `if` statement ensures that threads only process valid elements within the vector's bounds (`n`). If `idx` is within `n`, the thread performs the vector addition: `c[idx] = a[idx] + b[idx]`, where each thread computes one element of the sum.

```cuda
int main() {
```
This is the entry point of the host (CPU) program.

```cuda
    int num_elements = 1 << 20; // 1,048,576
```
Defines the number of elements in the vectors to be added. `1 << 20` is a bit shift operation equivalent to 2 raised to the power of 20, which is 1,048,576.

```cuda
    std::cout << "Vector Addition (N = "
              << num_elements << ")\n";
```
Prints the number of elements to the console.

```cuda
    size_t bytes = num_elements * sizeof(float);
```
Calculates the total memory in bytes required for each vector (number of elements multiplied by the size of a single float).

```cuda
    // Host vectors
    std::vector<float> h_a(num_elements);
    std::vector<float> h_b(num_elements);
    std::vector<float> h_c_cpu(num_elements);
    std::vector<float> h_c_gpu(num_elements);
```
Declares four `std::vector` objects on the host (CPU):
*   `h_a`, `h_b`: Input vectors.
*   `h_c_cpu`: To store the result of CPU vector addition (for verification).
*   `h_c_gpu`: To store the result copied back from the GPU.

```cuda
    // Initialize input vectors
    for (int i = 0; i < num_elements; i++) {
        h_a[i] = static_cast<float>(i);
        h_b[i] = static_cast<float>(2 * i);
    }
```
Initializes the host input vectors `h_a` and `h_b` with sample data.

```cuda
    // Device pointers
    float *d_a = nullptr;
    float *d_b = nullptr;
    float *d_c = nullptr;
```
Declares three pointers that will point to memory locations on the device (GPU). They are initialized to `nullptr`.

```cuda
    // Allocate GPU memory
    CUDA_CHECK(cudaMalloc((void**)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_c, bytes));
```
Allocates memory on the GPU for the input (`d_a`, `d_b`) and output (`d_c`) vectors. `cudaMalloc` is a CUDA API call for device memory allocation. `CUDA_CHECK` ensures error handling.

```cuda
    // Copy inputs to GPU
    CUDA_CHECK(cudaMemcpy(d_a, h_a.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, h_b.data(), bytes, cudaMemcpyHostToDevice));
```
Copies the initialized data from the host vectors (`h_a`, `h_b`) to their respective device memory locations (`d_a`, `d_b`). `cudaMemcpyHostToDevice` specifies the direction of the copy.

```cuda
    int threads_per_block = 256;
    int blocks = (num_elements + threads_per_block - 1) / threads_per_block;
```
Calculates the kernel launch configuration:
*   `threads_per_block`: The number of threads to execute in each thread block (a common value).
*   `blocks`: The total number of thread blocks needed. The formula ensures that all `num_elements` are covered, even if `num_elements` is not perfectly divisible by `threads_per_block` (ceiling division).

```cuda
    // CUDA events
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
```
Creates two CUDA events (`start` and `stop`). These events are used to precisely measure the execution time of the CUDA kernel on the GPU.

```cuda
    CUDA_CHECK(cudaEventRecord(start));
```
Records the `start` event, marking the beginning of the timed section.

```cuda
    add_kernel<<<blocks, threads_per_block>>>(d_a, d_b, d_c, num_elements);
```
This is the *kernel launch*! It invokes the `add_kernel` function on the GPU.
*   `<<<blocks, threads_per_block>>>`: This is the execution configuration, specifying `blocks` number of thread blocks, each containing `threads_per_block` threads.
*   `(d_a, d_b, d_c, num_elements)`: These are the arguments passed to the `add_kernel` function, which are pointers to device memory and the size `n`.

```cuda
    // Check kernel launch
    CUDA_CHECK(cudaGetLastError());
```
After launching a kernel, it's good practice to check for any asynchronous errors that might have occurred during the launch configuration or kernel execution setup. `cudaGetLastError()` retrieves the last CUDA error.

```cuda
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
```
Records the `stop` event and then `cudaEventSynchronize(stop)` waits until the `stop` event has been recorded (meaning all GPU work up to that point, including the kernel execution, has completed).

```cuda
    float gpu_time_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&gpu_time_ms, start, stop));
```
Calculates the elapsed time between the `start` and `stop` events in milliseconds and stores it in `gpu_time_ms`.

```cuda
    // Copy output back
    CUDA_CHECK(cudaMemcpy(h_c_gpu.data(), d_c, bytes,
                          cudaMemcpyDeviceToHost));
```
Copies the result of the vector addition from the device output array (`d_c`) back to the host output array (`h_c_gpu`). `cudaMemcpyDeviceToHost` specifies the direction.

```cuda
    // CPU timing
    auto cpu_start = std::chrono::high_resolution_clock::now();

    for (int i = 0; i < num_elements; i++)
        h_c_cpu[i] = h_a[i] + h_b[i];

    auto cpu_end = std::chrono::high_resolution_clock::now();

    std::chrono::duration<double, std::milli>
        cpu_time = cpu_end - cpu_start;
```
Performs the same vector addition on the CPU to get a baseline for performance comparison and for verification. It uses `std::chrono` for accurate CPU timing.

```cuda
    // Verify
    bool passed = true;

    for (int i = 0; i < num_elements; i++) {
        if (std::fabs(h_c_cpu[i] - h_c_gpu[i]) > 1e-5f) {
            passed = false;
            std::cout << "Mismatch at " << i << std::endl;
            break;
        }
    }
```
Compares the GPU result (`h_c_gpu`) with the CPU result (`h_c_cpu`) element by element. It checks for a small difference (floating-point tolerance `1e-5f`) due to potential precision variations. If a mismatch is found, `passed` is set to `false`, and the loop breaks.

```cuda
    std::cout << std::fixed << std::setprecision(3);
    std::cout << "GPU Kernel Time : "
              << gpu_time_ms << " ms\n";

    std::cout << "CPU Time        : "
              << cpu_time.count() << " ms\n";

    std::cout << "Verification: "
              << (passed ? "PASSED" : "FAILED")
              << std::endl;
```
Prints the GPU kernel execution time, CPU execution time, and the verification result (PASSED or FAILED) to the console, formatted to three decimal places.

```cuda
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
```
Frees the memory allocated on the device for `d_a`, `d_b`, and `d_c`. This is crucial to prevent memory leaks on the GPU.

```cuda
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
```
Destroys the CUDA events to release their resources.

```cuda
    return 0;
}
```
Indicates successful execution of the `main` function.


SyntaxError: unterminated string literal (detected at line 72) (3174602197.py, line 72)